## Goal of this notebook
Characterize the tandem repeat and repeat lengths and maybe create a bedfile 


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from natsort import natsorted
from natsort import index_natsorted
from matplotlib.patches import Patch
from Bio import SeqIO


In [ ]:


def parse_tandem_dat_file(file_path):
    # Initialize variables
    data = []
    current_sequence = None
    
    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            
            # Check for sequence header
            if line.startswith('Sequence:'):
                current_sequence = line.split(':')[-1].strip()
                continue
                
            # Skip empty lines or header lines
            if not line or line.startswith(('Tandem Repeats Finder', 'Parameters:', 'Sequence:', 'Version', 'Gary Benson', 'Boston University')):
                continue
                
            # Process data lines (they start with numbers)
            if line and line[0].isdigit() and current_sequence:
                # Split the line into parts
                parts = re.split(r'\s+', line)
                
                # Extract the relevant columns
                try:
                    start = int(parts[0])
                    end = int(parts[1])
                    period_size = int(parts[2])
                    copies_aligned = float(parts[3])
                    consensus_size = int(parts[4])
                    match_percent = float(parts[5])
                    indel_percent = float(parts[6])
                    alignment_score = float(parts[7])
                    
                    # The consensus and repeat sequences might be merged if they contain spaces
                    # So we need to handle them differently
                    consensus_seq = ' '.join(parts[13:-1]) if len(parts) > 14 else parts[13]
                    repeat_seq = parts[-1] if len(parts) > 14 else ''
                    
                    data.append({
                        'sequence': current_sequence,
                        'start': start,
                        'end': end,
                        'period_size': period_size,
                        'copies_aligned': copies_aligned,
                        'consensus_size': consensus_size,
                        'match_percent': match_percent,
                        'indel_percent': indel_percent,
                        'alignment_score': alignment_score,
                        'consensus_sequence': consensus_seq,
                        'repeat_sequence': repeat_seq
                    })
                except (IndexError, ValueError) as e:
                    print(f"Error processing line: {line}")
                    print(f"Error: {e}")
                    continue
                    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Reorder columns as requested
    columns = ['sequence', 'start', 'end', 'period_size', 'alignment_score']
    # Add any additional columns you want to keep
    additional_cols = [col for col in df.columns if col not in columns]
    df = df[columns + additional_cols]
    
    return df

# Example usage:
# df = parse_tandem_dat_file('your_file.dat')
# print(df.head())

In [ ]:


def parse_tandem_dat_line(line, current_sequence):
    """Parse a single line of tandem repeat data."""
    parts = re.split(r'\s+', line.strip())
    
    # Extract the fixed-width columns (first 13 columns)
    try:
        data = {
            'sequence': current_sequence,
            'start': int(parts[0]),
            'end': int(parts[1]),
            'period_size': int(parts[2]),
            'copies_aligned': float(parts[3]),
            'consensus_size': int(parts[4]),
            'match_percent': float(parts[5]),
            'indel_percent': float(parts[6]),
            'alignment_score': float(parts[7]),
            'percent_A': float(parts[8]),
            'percent_C': float(parts[9]),
            'percent_G': float(parts[10]),
            'percent_T': float(parts[11]),
            'entropy': float(parts[12]),
        }
        
        # Handle consensus and repeat sequences (may contain spaces)
        if len(parts) > 13:
            data['consensus_sequence'] = ' '.join(parts[13:-1]) if len(parts) > 14 else parts[13]
            data['repeat_sequence'] = parts[-1] if len(parts) > 14 else ''
        else:
            data['consensus_sequence'] = ''
            data['repeat_sequence'] = ''
            
        return data
        
    except (IndexError, ValueError) as e:
        print(f"Error processing line: {line}")
        print(f"Error: {e}")
        return None

def parse_tandem_dat_file(file_path):
    """Parse a single .dat file and return a DataFrame."""
    data = []
    current_sequence = None
    
    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            
            if line.startswith('Sequence:'):
                current_sequence = line.split(':')[-1].strip()
            elif line and line[0].isdigit() and current_sequence:
                parsed_data = parse_tandem_dat_line(line, current_sequence)
                if parsed_data:
                    data.append(parsed_data)
    
    return pd.DataFrame(data)

def parse_dat_files_in_directory(directory):
    """Parse all .dat files in a directory and return a combined DataFrame."""
    all_dfs = []
    
    for filename in os.listdir(directory):
        if filename.endswith('.dat'):
            file_path = os.path.join(directory, filename)
            print(f"Processing file: {filename}")
            df = parse_tandem_dat_file(file_path)
            all_dfs.append(df)
    
    if not all_dfs:
        print(f"No .dat files found in directory: {directory}")
        return pd.DataFrame()
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    return combined_df


def get_contig_lengths(fasta_path):
    """Returns a dictionary of {full_header: length} from a FASTA file."""
    contig_lengths = {}
    with open(fasta_path, "r") as fasta_file:
        for record in SeqIO.parse(fasta_file, "fasta"):
            # Use record.description to get the full header line (after '>')
            full_header = record.description
            contig_lengths[full_header] = len(record.seq)
    return contig_lengths




In [ ]:
fa_file="/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/data/GRCm39_genome/GCF_000001635.27_GRCm39_genomic.fna"
input_path="/tscc/lustre/ddn/scratch/jhc103/trf-output-mouse"

# List of 30 distinct colors (RGB tuples)
discrete_colors_30 = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
    '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5',
    '#393b79', '#637939', '#8c6d31', '#843c39', '#7b4173',
    '#5254a3', '#8ca252', '#bd9e39', '#ad494a', 'black', 
]

In [ ]:
contig_lengths = get_contig_lengths(fa_file)
repeat_df=parse_dat_files_in_directory(input_path)

In [ ]:
set(repeat_df['sequence'])

In [ ]:
repeat_df=repeat_df[~repeat_df['sequence'].str.contains('genomic scaffold', na=False)]


In [ ]:
repeat_df['seq_length'] = repeat_df["sequence"].map(contig_lengths)

In [ ]:
# Extract chromosome number (1-22, X, Y) and format as 'chrN'
repeat_df['sequence_orig']=repeat_df['sequence']
repeat_df['sequence']=repeat_df['sequence_orig'].str.extract(r'chromosome (\d+|X|Y)')[0].apply(lambda x: f'chr{x}')

print(repeat_df['sequence'].unique())
# print(df['Chromosome_short'])

In [ ]:
repeat_df.to_csv("repeat_df_mouse.tsv", sep='\t',index=False)

In [ ]:
repeat_df=pd.read_csv("repeat_df_mouse.tsv", sep='\t')

In [ ]:
## Create length column
repeat_df["length"]=repeat_df["end"]-repeat_df["start"]

repeat_df["repeat_point"]=repeat_df["start"]+(repeat_df["end"]-repeat_df["start"])/2


## Create chromosome column
repeat_df['chromosome'] = repeat_df['sequence'].apply(
    lambda x: 'seq_additional' if str(x).lower().startswith('seq') else x
)


In [ ]:
# Compute chromosome_midpoint
repeat_df['chromosome_midpoint'] = repeat_df['seq_length']/ 2

In [ ]:

## Relative position to center
repeat_df["repeat_point_relative"] = repeat_df["repeat_point"]-repeat_df["chromosome_midpoint"]
# Relative position to center 
repeat_df["repeat_point_relative_perc"] = np.where(
    repeat_df["repeat_point_relative"] == 0,
    0,
    (repeat_df["repeat_point_relative"] / repeat_df["seq_length"]) * 100
)

In [ ]:
# Natural sort the DataFrame (if needed)
repeat_df = repeat_df.sort_values(by='chromosome', key=lambda x: np.argsort(index_natsorted(repeat_df["chromosome"])))


In [ ]:
# repeat_df['chromosome'] = repeat_df['sequence'].apply(
#     lambda x: 'seq_additional' if str(x).lower().startswith('seq') else x
# )

# # chromosome_order = [f'chr{i}' for i in range(1, 28)] + ['chrX', 'chrY', 'seq_additional']
# # Convert 'chromosome' to a categorical variable with this order
# repeat_df['chromosome'] = pd.Categorical(
#     repeat_df['chromosome'], 
#     categories=chromosome_order, 
#     ordered=True
# )
# # Sort the DataFrame (if needed)
# repeat_df = repeat_df.sort_values('chromosome')

## Put color 
color_mapping = {chrom: discrete_colors_30[i] for i, chrom in enumerate(repeat_df['chromosome'].unique())}

repeat_df['color'] = repeat_df['chromosome'].map(color_mapping)

In [ ]:
plt.hist(repeat_df["length"], bins=100)

In [ ]:

plt.figure(figsize=(10, 6))
plt.hist(repeat_df['length'], bins=100, edgecolor='black', log=True)

# # Disable scientific notation on x-axis
# plt.ticklabel_format(axis='x', style='plain')  # or use 'plain' instead of 'sci'
# plt.title('Histogram of Alignment Scores (No Scientific Notation)')
# plt.xlabel('Alignment Score')
# plt.ylabel('Frequency')
# plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=repeat_df, 
             x='length', 
             bins=30, 
             edgecolor='black',
             log_scale=True,  # Log scale for both axes
             multiple='stack',  # Stack bars
             palette='viridis')  

In [ ]:
# Create legend handles (dummy patches)
legend_handles = [
    Patch(color=color, label=chr_name) 
    for chr_name, color in color_mapping.items()
]

# Create a figure (empty plot) and add only the legend
plt.figure(figsize=(4, 2))  # Small figure to fit just the legend
plt.legend(handles=legend_handles, ncol=3, title='Chromosomes', loc='center')

# Remove axes (optional)
plt.axis('off')

plt.tight_layout()  # Prevent text cutoff
plt.show()

In [ ]:

# Assuming `repeat_df` and `color_mapping` are defined

plt.figure(figsize=(12, 6))

# Create the histogram with Seaborn (disable auto-legend)
ax = sns.histplot(data=repeat_df, 
                 x='length', 
                 bins=30, 
                 hue='chromosome', 
                 log_scale=True,
                 multiple='stack', 
                 palette=color_mapping,
                 legend=False)  # <- Critical: Disable Seaborn's legend

# Custom legend using Patch (replace internal legend)
legend_handles = [
    Patch(color=color, label=chr_name) 
    for chr_name, color in color_mapping.items()
]

# Add the legend to the plot
plt.legend(
    handles=legend_handles,
    ncol=3,                # 3 columns (adjust as needed)
    title='Chromosomes',
    bbox_to_anchor=(1, 1),  # Place outside top-right
    loc='upper right'
)

plt.tight_layout()
plt.xlabel('Total Repeat Length (bp)')  # Descriptive x-axis label
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Create the histogram with Seaborn (disable auto-legend)
ax = sns.histplot(data=repeat_df[repeat_df["length"]>1000], 
                 x='length', 
                 bins=30, 
                 hue='chromosome', 
                 log_scale=True,
                 multiple='stack', 
                 palette=color_mapping,
                 legend=False)  # <- Critical: Disable Seaborn's legend

# Custom legend using Patch (replace internal legend)
legend_handles = [
    Patch(color=color, label=chr_name) 
    for chr_name, color in color_mapping.items()
]

# Add the legend to the plot
plt.legend(
    handles=legend_handles,
    ncol=3,                # 3 columns (adjust as needed)
    title='Chromosomes',
    bbox_to_anchor=(1, 1),  # Place outside top-right
    loc='upper right'
)

plt.tight_layout()
plt.xlabel('Total Repeat Length (bp)')  # Descriptive x-axis label
plt.show()

In [ ]:
shuffled_df = repeat_df.sample(frac=1, random_state=28)  # frac=1 means "shuffle all rows"


In [ ]:
## Plot the repeats as scatter plots 
## x - length 
## repeats 
## Size - period_size
## Color - chromosome

plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df,
               x="length",
               y="copies_aligned",
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               size="period_size",
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
ax.set_xscale('log')
ax.set_yscale('log')
plt.xlabel('Total tandem repeat Length (bp)')  # Descriptive x-axis label
plt.ylabel('Number of repeats in a tandem repeat')  # Descriptive x-axis label

plt.legend(ncol=3,  loc='lower right',bbox_to_anchor=(1,0))  # This sets 2 columns and a title


In [ ]:
## Plot the repeats as scatter plots 
## x - length 
## repeats 
## Size - period_size
## Color - chromosome

plt.figure(figsize=(8, 8))
sns.scatterplot(data=shuffled_df,
               x="length",
               y="period_size",
               hue="chromosome",
                size="length",
               sizes=(2, 300),  
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
ax.set_xscale('log')
# ax.set_yscale('log')
plt.xlabel('Total tandem repeat Length (bp)',fontsize=12)  # Descriptive x-axis label
plt.ylabel('1 repeat length (bp)',fontsize=12)  # Descriptive x-axis label
horizontal_y = 120  # Example value - change this to your desired y-position
# ax.axhline(y=horizontal_y, color='gray', linestyle='--', linewidth=1, alpha=0.5)
# horizontal_y = 145  # Example value - change this to your desired y-position
ax.axhline(y=horizontal_y, color='gray', linestyle='--', linewidth=1, alpha=0.5)
horizontal_y = 234  # Example value - change this to your desired y-position
ax.axhline(y=horizontal_y, color='gray', linestyle='--', linewidth=1, alpha=0.5)
# plt.legend(ncol=3, loc="upper left",bbox_to_anchor=(0, 1))  # This sets 2 columns and a title
# Modify legend with fontsize control
plt.legend(ncol=3, 
           loc="upper left",
           bbox_to_anchor=(0, 1),
           fontsize=6)  # Adjust fontsize here (default is usually 10)

In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df,
               x="copies_aligned",
               y="period_size",
               hue="chromosome",
                size="length",
               sizes=(2, 300),  
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
ax.set_xscale('log')
# ax.set_yscale('log')
plt.xlabel('Number of repeats in a tandem repeat')  # Descriptive x-axis label
plt.ylabel('1 repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=3, bbox_to_anchor=(1.001, 1))  # This sets 2 columns and a title


In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df,
               y="length",
               x="repeat_point_relative",
               size="period_size",
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
# ax.set_xscale('log')
ax.set_yscale('log')
plt.xlabel('Centered repeat position')  # Descriptive x-axis label
plt.ylabel('Total tandem repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=1, bbox_to_anchor=(1.001, 1))  # This sets 2 columns and a title


In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df,
               y="length",
               x="repeat_point_relative_perc",
               size="period_size",
               sizes=(0.2, 100),  # Adjust these values for more dramatic size differences
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
# ax.set_xscale('log')
ax.set_yscale('log')
# --- X-axis ticks: Every 10 units from -50 to 50 ---
x_min, x_max = -50, 50  # Your specified range
x_ticks = np.arange(x_min, x_max + 10, 10)  # +10 to include 50

ax.set_xticks(x_ticks)
ax.set_xticklabels([f"{x:.0f}" for x in x_ticks])  # No decimals for integers

# --- Labels/legend ---
plt.xlabel('Relative centered repeat position (%)')
plt.ylabel('Total tandem repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=1, loc="upper left",bbox_to_anchor=(1.01, 1))  # This sets 2 columns and a title


In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df[shuffled_df["period_size"]>250],
               y="length",
               x="repeat_point_relative_perc",
               size="period_size",
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
# ax.set_xscale('log')
ax.set_yscale('log')
# --- X-axis ticks: Every 10 units from -50 to 50 ---
x_min, x_max = -50, 50  # Your specified range
x_ticks = np.arange(x_min, x_max + 10, 10)  # +10 to include 50

ax.set_xticks(x_ticks)
ax.set_xticklabels([f"{x:.0f}" for x in x_ticks])  # No decimals for integers

# --- Labels/legend ---
plt.xlabel('Relative centered repeat position (%)')
plt.ylabel('Total tandem repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=1, bbox_to_anchor=(1.001, 1))  # This sets 2 columns and a title


In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df[shuffled_df["period_size"]>349],
               y="length",
               x="repeat_point_relative_perc",
               size="period_size",
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
# ax.set_xscale('log')
ax.set_yscale('log')
# --- X-axis ticks: Every 10 units from -50 to 50 ---
x_min, x_max = -50, 50  # Your specified range
x_ticks = np.arange(x_min, x_max + 10, 10)  # +10 to include 50

ax.set_xticks(x_ticks)
ax.set_xticklabels([f"{x:.0f}" for x in x_ticks])  # No decimals for integers

# --- Labels/legend ---
plt.xlabel('Relative centered repeat position (%)')
plt.ylabel('Total tandem repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=1, bbox_to_anchor=(1.001, 1))  # This sets 2 columns and a title


In [ ]:
plt.figure(figsize=(8, 8))
sns.scatterplot(data=shuffled_df[shuffled_df["length"]>0],
               y="period_size",
               x="repeat_point_relative_perc",
               size="length",
               sizes=(2, 300),  # Adjust these values for more dramatic size differences
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
# --- X-axis ticks: Every 10 units from -50 to 50 ---
x_min, x_max = -50, 50  # Your specified range
x_ticks = np.arange(x_min, x_max + 10, 10)  # +10 to include 50

ax.set_xticks(x_ticks)
ax.set_xticklabels([f"{x:.0f}" for x in x_ticks])  # No decimals for integers

# --- Labels/legend ---
plt.xlabel('Relative centered repeat position (%)',fontsize=12)
plt.ylabel('1 repeat length (bp)',fontsize=12)  # Descriptive x-axis label

# plt.legend(ncol=1, loc="upper left",bbox_to_anchor=(1.01, 1))  # This sets 2 columns and a title
plt.legend(ncol=1, 
           bbox_to_anchor=(1.01, 1),
           loc="upper left"
           fontsize=6)  # Adjust fontsize here (default is usually 10)

In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df[shuffled_df["length"]>0],
               y="period_size",
               x="repeat_point_relative_perc",
               # size="period_size",
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()

# --- X-axis ticks: Every 10 units from -50 to 50 ---
x_min, x_max = -50, 50  # Your specified range
x_ticks = np.arange(x_min, x_max + 10, 10)  # +10 to include 50

ax.set_xticks(x_ticks)
ax.set_xticklabels([f"{x:.0f}" for x in x_ticks])  # No decimals for integers

# --- Labels/legend ---
plt.xlabel('Relative centered repeat position (%)')
plt.ylabel('1 repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=1, loc="upper left",bbox_to_anchor=(1.01, 1))  # This sets 2 columns and a title


In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=shuffled_df,
               y="period_size",
               x="repeat_point_relative",
               size="period_size",
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
# --- X-axis ticks: Every 10 units from -50 to 50 ---
x_min, x_max = -50, 50  # Your specified range
x_ticks = np.arange(x_min, x_max + 10, 10)  # +10 to include 50

ax.set_xticks(x_ticks)
ax.set_xticklabels([f"{x:.0f}" for x in x_ticks])  # No decimals for integers

# --- Labels/legend ---
plt.xlabel('Relative centered repeat position (%)')
plt.ylabel('1 repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=3, bbox_to_anchor=(1.001, 1))  # This sets 2 columns and a title


In [ ]:

## Check out the data 
# test = repeat_df[(repeat_df["period_size"]>192) & (repeat_df["period_size"]<197)]
test = repeat_df[repeat_df["length"]>10000]

In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=test,
               x="copies_aligned",
               y="period_size",
               hue="chromosome",
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
ax.set_xscale('log')
# ax.set_yscale('log')
plt.xlabel('Number of repeats in a tandem repeat')  # Descriptive x-axis label
plt.ylabel('1 repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=3, bbox_to_anchor=(1.001, 1))  # This sets 2 columns and a title


In [ ]:
# Create IGV-compatible BED file
bed_df = pd.DataFrame({
    'chrom': repeat_df['chromosome'],
    'start': repeat_df['start'] - 1,  # Convert to 0-based for BED
    'end': repeat_df['end'],         # BED end remains 1-based
    'name': 'TRF_' + repeat_df['period_size'].astype(str) + 'bp',
    'score': (repeat_df['alignment_score'] / repeat_df['alignment_score'].max() * 1000).astype(int),  # Scale score to 0-1000
    'strand': '.',
    'thickStart': repeat_df['start'] - 1,  # For visualization
    'thickEnd': repeat_df['end'],          # For visualization
    'itemRgb': repeat_df['color']         # Use your color column
})

## Check out the data 
subset_bed = bed_df.loc[test.index]
subset_bed.to_csv("all_repeat_igv_mouse.bed", sep="\t", header=False, index=False)

# out_df = bed_df[(bed_df["period_size"]<351) & (bed_df["period_size"]>347)]

## Subset and save

In [ ]:
# repeat_int=shuffled_df[(shuffled_df["length"]>1000) & (shuffled_df["period_size"]<390) & (shuffled_df["period_size"]>387)]
repeat_int=shuffled_df[(shuffled_df["length"]>1000) & (shuffled_df["period_size"]<172) & (shuffled_df["period_size"]>168)]


In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=repeat_int,
               x="copies_aligned",
               y="period_size",
               hue="chromosome",
                size="length",
               sizes=(2, 300),  
               hue_order=list(color_mapping.keys()),
               palette=color_mapping)
# Get the current axes
ax = plt.gca()
ax.set_xscale('log')
# ax.set_yscale('log')
plt.xlabel('Number of repeats in a tandem repeat')  # Descriptive x-axis label
plt.ylabel('1 repeat length (bp)')  # Descriptive x-axis label

plt.legend(ncol=3, bbox_to_anchor=(1.001, 1))  # This sets 2 columns and a title


In [ ]:
# repeat_int.to_csv("./389peak_repeat.tsv",sep='\t')
# repeat_int.to_csv("./349peak_repeat.tsv",sep='\t')
repeat_int.to_csv("./171peak_repeat_human.tsv",sep='\t')

In [ ]:

new_df = pd.DataFrame({
    'repeat_sequence': repeat_int['repeat_sequence'],
    'sequence_start_end': repeat_int['sequence'] + '_' + repeat_int['start'].astype(str) + '_' + repeat_int['end'].astype(str)
})
new_df

In [ ]:
# with open("389peak_repeat.fasta", "w") as f:
# with open("349peak_repeat.fasta", "w") as f:
with open("171peak_repeat_human.fasta", "w") as f:
    for idx, row in new_df.iterrows():
        f.write(f">{row['sequence_start_end']}\n")
        f.write(f"{row['repeat_sequence']}\n")